# 03 — Action repeat (HOOK B)

The temporal intervention: call the model **half as often** and hold each
action for two environment steps instead of one.

```python
actions = np.repeat(actions, 2, axis=0)
```

That single line is the whole mechanism. What follows is why it goes where
it goes, how it differs from chunk execution, and how to measure it without
reporting the wrong number.

## Action repeat vs chunk execution — these are different interventions

Both reduce the number of model calls. They are not interchangeable.

| | action repeat 2 | chunk-exec 2 |
|---|---|---|
| what gets executed | the **same action, copied** | **two different actions** the model actually predicted |
| requires | nothing | the policy must emit multiple actions per call |
| information lost | yes — motion becomes stepwise | comparatively little |
| applies to OpenVLA | ✅ | ❌ **impossible** |

**OpenVLA emits exactly one action per call**, so there is no chunk to
truncate — chunk-exec is not a slower or worse option for it, it simply
does not exist. Action repeat is therefore the only temporal intervention
that runs *identically* on every backbone, which is why it is the one in
the comparison grid.

That is itself a finding worth stating in the analysis: the temporal axis
is really two mechanisms — one universal but lossy, one lossless but
requiring native action chunking — and a backbone that cannot chunk is
forced onto the worse one.

For reference, on SimplerEnv Bridge chunk-exec at k=2 was SpatialVLA's
strongest result (32.3% → 45.9%, at 1.9× faster) while it *cost* UniVLA
12.5 points. Same intervention, opposite sign.

## HOOK B — where this goes

On the **action array the policy returned, before it reaches `env.step`.**

```
actions = policy.step(image, instruction)   # (T, action_dim)
actions = np.repeat(actions, 2, axis=0)     ← HERE   -> (2T, action_dim)
for row in actions:
    env.step(row)
```

### Order matters if combined with chunk truncation

If a run uses both, truncate **first**, then repeat:

```python
actions = actions[:k]                       # chunk-exec
actions = np.repeat(actions, r, axis=0)     # action repeat
```

Repeating first and then truncating silently produces a different
condition — with `k=2, r=2` it would execute the first action twice and
nothing else, rather than two actions twice each.

In [ ]:
import numpy as np


def apply_action_repeat(actions, repeat):
    """Hold each action for `repeat` consecutive environment steps.

    np.repeat (not np.tile): repeat=2 on [a, b, c] gives [a, a, b, b, c, c],
    whereas tile would give [a, b, c, a, b, c] — a completely different
    trajectory that would still run and still produce a number.
    """
    actions = np.asarray(actions)
    if repeat <= 1:
        return actions
    return np.repeat(actions, int(repeat), axis=0)


def make_action_repeat_hook(repeat=2, exec_chunk=0):
    """Returns an action_fn for `run_episode` in notebook 01."""

    def action_fn(actions, state):
        if exec_chunk > 0:
            actions = actions[:exec_chunk]      # truncate first
        return apply_action_repeat(actions, repeat)

    return action_fn


# usage with the loop from 01:
#   run_episode(env, policy, instruction,
#               action_fn=make_action_repeat_hook(repeat=2))

In [ ]:
demo = np.array([[0.1, 0.0], [0.2, 0.0], [0.3, 0.0]])
print("original      :", demo[:, 0].tolist())
print("repeat 2      :", apply_action_repeat(demo, 2)[:, 0].tolist())
print("tile (WRONG)  :", np.tile(demo, (2, 1))[:, 0].tolist())

## What it costs the trajectory

A repeated action doubles the displacement commanded before the policy sees
a new frame. In a delta-pose controller that means the arm travels twice as
far open-loop between corrections. The failure mode is therefore *overshoot
on approach and imprecision at contact*, not a uniform degradation — which
is why it hurts tasks needing fine placement far more than coarse reaching.

In [ ]:
# A policy tracking a target with proportional control, under repeat=1 vs 2.
# The gain is stable when each action is applied once; repeating it doubles
# the effective gain, which is what pushes the loop past 1.0 and overshoots.
def simulate(repeat, steps=40, gain=0.6, target=1.0):
    pos, trace = 0.0, []
    t = 0
    while t < steps:
        action = gain * (target - pos)          # one model call
        for _ in range(repeat):                 # executed `repeat` times
            pos += action
            trace.append(pos)
            t += 1
            if t >= steps:
                break
    return trace

a, b = simulate(1), simulate(2)
print(f"repeat=1  final {a[-1]:.3f}   max overshoot {max(a) - 1.0:+.3f}")
print(f"repeat=2  final {b[-1]:.3f}   max overshoot {max(b) - 1.0:+.3f}")
print("\nBoth converge here because the target does not move. On a real "
      "task\nthe overshoot lands the gripper past the object, and the "
      "correction\narrives one full call late.")

try:
    import matplotlib.pyplot as plt
    plt.figure(figsize=(7, 3.2))
    plt.axhline(1.0, color="gray", ls="--", lw=1, label="target")
    plt.plot(a, label="repeat=1")
    plt.plot(b, label="repeat=2")
    plt.xlabel("environment step"); plt.ylabel("position")
    plt.legend(); plt.tight_layout(); plt.show()
except ImportError:
    pass

## Measuring it correctly

**The cost of one model call does not change.** What halves is the number
of calls. So the per-call figure is identical to baseline and reporting it
alone would suggest the intervention did nothing.

Report either:

* **calls per episode** (halved), or
* **ms per environment step** = `ms_per_call × calls / env_steps` (halved),

and say which. The `run_episode` in notebook 01 returns both
`ms_per_call` and `ms_per_env_step` for exactly this reason.

One more asymmetry worth knowing: a policy that already emits a chunk of 10
and executes all of them is *already* amortised 10× per call. Applying
repeat=2 on top pushes it to 20 environment steps of open-loop execution
between observations. That is why the same repeat=2 that cost OpenVLA 8
points (within noise) cost UniVLA 68.

In [ ]:
# Confirm the two figures move as expected, using the stubs from 01.
# (Paste the StubEnv/StubPolicy/run_episode cells from 01 first.)
#
# base = run_episode(StubEnv(), StubPolicy(chunk=4), "task")
# rep  = run_episode(StubEnv(), StubPolicy(chunk=4), "task",
#                    action_fn=make_action_repeat_hook(2))
# print("calls:", base["model_calls"], "->", rep["model_calls"])
# assert rep["model_calls"] < base["model_calls"]